In [0]:
# =============================== 
# Load Pipeline Functions 
# ===============================
%run ../Src/FaersPipeline
%run ../Src/DataContractValidator

contract = load_contract("../Src/Contracts.yaml")

# =============================== 
# Imports & Setup 
# ===============================
from pyspark.sql.functions import *
from pyspark.sql.functions import col
from collections import Counter

# Create schemas if they don't exist
spark.sql("DROP SCHEMA IF EXISTS bronze CASCADE")
spark.sql("DROP SCHEMA IF EXISTS silver CASCADE")
spark.sql("DROP SCHEMA IF EXISTS gold CASCADE")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print("Schemas created (bronze, silver, gold)")

# =============================== 
# Helper Functions 
# ===============================
def normalize_columns(df):
    """
    Normalize all column names to lowercase so Spark/Delta does not
    treat Quarter vs quarter as conflicting columns.
    """
    return df.toDF(*[c.lower() for c in df.columns])

def print_duplicate_columns(df, df_name):
    duplicates = [c for c, n in Counter(df.columns).items() if n > 1]
    print(f"{df_name} duplicate columns: {duplicates if duplicates else 'None'}")

# =============================== 
# Bronze Layer - Load FAERS Data 
# ===============================

# 2023
Demographics2023 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_adverse_events_reporting_system_demographics_2023"
)
Drug2023 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_adverse_events_reporting_system_drug_2023"
)
Reaction2023 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_adverse_events_reporting_system_drug_reaction_2023"
)
Outcome2023 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_drug_adverse_events_reporting_system_faers_2023."
    "fda_adverse_events_reporting_system_patient_outcome_2023"
)

# 2022
Demographics2022 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_adverse_events_reporting_system_demographics_2022"
)
Drug2022 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_adverse_events_reporting_system_drug_2022"
)
Reaction2022 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_adverse_events_reporting_system_drug_reaction_2022"
)
Outcome2022 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_drug_adverse_events_reporting_system_faers_2022."
    "fda_adverse_events_reporting_system_patient_outcome_2022"
)

# 2021
Demographics2021 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_adverse_events_reporting_system_demographics_2021"
)
Drug2021 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_adverse_events_reporting_system_drug_2021"
)
Reaction2021 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_adverse_events_reporting_system_drug_reaction_2021"
)
Outcome2021 = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_drug_adverse_events_reporting_system_faers_2021."
    "fda_adverse_events_reporting_system_patient_outcome_2021"
)

# Normalize Column Names
Demographics2021 = normalize_columns(Demographics2021)
Demographics2022 = normalize_columns(Demographics2022)
Demographics2023 = normalize_columns(Demographics2023)
Drug2021 = normalize_columns(Drug2021)
Drug2022 = normalize_columns(Drug2022)
Drug2023 = normalize_columns(Drug2023)
Reaction2021 = normalize_columns(Reaction2021)
Reaction2022 = normalize_columns(Reaction2022)
Reaction2023 = normalize_columns(Reaction2023)
Outcome2021 = normalize_columns(Outcome2021)
Outcome2022 = normalize_columns(Outcome2022)
Outcome2023 = normalize_columns(Outcome2023)
print("Column names normalized to lowercase")

# Fix data type
Drug2021_fixed = (
    Drug2021
    .withColumn("Cumulative_Dose_to_First_Reaction", col("Cumulative_Dose_to_First_Reaction").cast("string"))
    .withColumn("Drug_Expiration_Date", col("Drug_Expiration_Date").cast("string"))
)
Drug2022_fixed = (
    Drug2022
    .withColumn("Cumulative_Dose_to_First_Reaction", col("Cumulative_Dose_to_First_Reaction").cast("string"))
    .withColumn("Drug_Expiration_Date", col("Drug_Expiration_Date").cast("string"))
)
Drug2023_fixed = (
    Drug2023
    .withColumn("Cumulative_Dose_to_First_Reaction", col("Cumulative_Dose_to_First_Reaction").cast("string"))
    .withColumn("Drug_Expiration_Date", col("Drug_Expiration_Date").cast("string"))
)

# Combine 3 years into single tables
Demographics = Demographics2021.unionByName(Demographics2022).unionByName(Demographics2023)
Drug = (Drug2021_fixed.unionByName(Drug2022_fixed, allowMissingColumns=True)
        .unionByName(Drug2023_fixed, allowMissingColumns=True))
Reaction = Reaction2021.unionByName(Reaction2022).unionByName(Reaction2023)
Outcome = Outcome2021.unionByName(Outcome2022).unionByName(Outcome2023)

# Keep only the latest version of each record in Demographics
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, col

windowSpec = Window.partitionBy("primary_id").orderBy(desc("case_version_number"))
Demographics = (Demographics
                .withColumn("row_num", row_number().over(windowSpec))
                .filter(col("row_num") == 1)
                .drop("row_num"))

# Save combined tables to bronze
Demographics.write.format("delta").mode("overwrite").saveAsTable("bronze.Demographics")
Drug.write.format("delta").mode("overwrite").saveAsTable("bronze.Drug")
Reaction.write.format("delta").mode("overwrite").saveAsTable("bronze.Reaction")
Outcome.write.format("delta").mode("overwrite").saveAsTable("bronze.Outcome")
print("Combined bronze tables created")

# =============================== 
# Run Data Contracts 
# ===============================
validate_table_contract(Demographics, "demographics", contract)
validate_table_contract(Drug, "drug", contract)
validate_table_contract(Reaction, "reaction", contract)
validate_table_contract(Outcome, "outcome", contract)
print("All data contracts passed")

# =============================== 
# Silver Layer 
# ===============================
silver_tables = create_silver_tables(Demographics, Drug, Reaction, Outcome)

# Demographics
silver_tables["demographics"].write.format("delta").mode("overwrite").saveAsTable("silver.demographics")
# Drugs
silver_tables["drugs"].write.format("delta").mode("overwrite").saveAsTable("silver.drugs")
# Reactions
silver_tables["reactions"].write.format("delta").mode("overwrite").saveAsTable("silver.reactions")
# Outcomes
silver_tables["outcomes"].write.format("delta").mode("overwrite").saveAsTable("silver.outcomes")
print("Silver tables created successfully")

# =============================== 
# Gold Layer 
# ===============================

# Load Silver tables from catalog
silver_demographics = spark.table("silver.demographics")
silver_drugs = spark.table("silver.drugs")
silver_reactions = spark.table("silver.reactions")
silver_outcomes = spark.table("silver.outcomes")

# Drug Adverse Event Counts Over Time
from pyspark.sql.functions import countDistinct

gold_drug_event_counts = (
    silver_drugs
    .join(silver_reactions, ["primary_id", "case_id", "year"], "left")
    .groupBy("year", "drug_name")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "year")
)
gold_drug_event_counts.write.format("delta").mode("overwrite").saveAsTable("gold.drug_event_counts")

# Drug Reaction Trends
from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

drug_reaction_counts = (
    silver_drugs
    .join(silver_reactions, ["primary_id", "case_id", "year"], "left")
    .groupBy("year", "drug_name", "preferred_term_for_event")
    .agg(countDistinct("primary_id").alias("num_reports"))
)

windowSpec = Window.partitionBy("year", "drug_name").orderBy(col("num_reports").desc())
gold_drug_reaction_trends = (
    drug_reaction_counts
    .withColumn("rank", row_number().over(windowSpec))
    .filter(col("rank") <= 10)
)
gold_drug_reaction_trends.write.format("delta").mode("overwrite").saveAsTable("gold.drug_reaction_trends")

# Severe Outcomes per Drug
severe_codes = ["DE", "HO", "LT"]
gold_drug_severe_outcomes = (
    silver_drugs
    .join(silver_outcomes, ["primary_id", "case_id", "year"], "left")
    .filter(col("patient_outcome").isin(severe_codes))
    .groupBy("year", "drug_name", "patient_outcome")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "year", "patient_outcome")
)
gold_drug_severe_outcomes.write.format("delta").mode("overwrite").saveAsTable("gold.drug_severe_outcomes")

# Patient Demographics per Drug
gold_drug_patient_groups = (
    silver_drugs
    .join(silver_demographics, ["primary_id", "case_id", "year"], "left")
    .groupBy("year", "drug_name", "age_group_readable", "gender_of_patient")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "year", "age_group_readable", "gender_of_patient")
)
gold_drug_patient_groups.write.format("delta").mode("overwrite").saveAsTable("gold.drug_patient_groups")

# =============================== 
# Show Results 
# ===============================
print("Drug Adverse Event Counts Over Time")
display(spark.table("gold.drug_event_counts"))

print("Drug Reaction Trends (Top 10 Reactions)")
display(spark.table("gold.drug_reaction_trends"))

print("Severe Outcomes per Drug")
display(spark.table("gold.drug_severe_outcomes"))

print("Patient Demographics per Drug")
display(spark.table("gold.drug_patient_groups"))

In [0]:
base_path = "/Workspace/Users/joble.thomas789@gmail.com/Project/DrugSafetyMonitoring/Resources/Contracts.yaml"

for prefix in ["file:", "/dbfs", ""]:  # test variants
    test_path = prefix + base_path if prefix else base_path
    try:
        with open(test_path, "r") as f:
            content = f.read(100)  # peek first 100 chars
            print(f"SUCCESS with prefix '{prefix}': {content.strip()[:50]}...")
        break
    except Exception as e:
        print(f"Failed with '{prefix}': {str(e)}")